In [1]:
import torch, transformers
import json
from copy import deepcopy

model_8B_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tok = transformers.AutoTokenizer.from_pretrained(model_8B_id)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_8B_id, dtype=torch.float16, device_map="cuda"
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
def get_test_data(path):
    examples = []
    # "D:\\GeoTKG\\cleandata\\tie\\test.json"
    with open(path, "r") as f:
        examples=[json.loads(line) for line in f]
    return examples

def get_ner_prompt(text, method):
    if method == "event and time":
        tags = "EVENT, DATE, TIME, DURATION, SET"
    elif method == "geoscience":
        tags = "LOCATION, MINERAL, ORE_DEPOSIT, ROCK, STRAT, TIMESCALE"
    sample = " ".join([wrd for sent in text for wrd in sent])
    sys = f'''You are a Named Entity Recognition (NER) system for tagging {method} entities. Identify and classify entities in the text based on the entity types: {tags}. 
            Each entity should be represented as a tuple (entity surface text, type) in valid JSON format.
            Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".'''
    
    user = f'''Extract entities from the following text: {sample}'''
    messages = [
        {"role": "system", "content": sys},
        {"role": "user", "content": user}
    ]
    return messages

def get_norm_prompt(text, dct):
    sys = '''
        You are a time normalization system. Normalize the time expressions that have been tagged in the text using the document creation time as anchor for calendar time.
        Each time mention should be represented as a tuple: [surface text, normalized value].
        Calendar time expressions should be in ISO 8601 format (YYYY-MM-DD).
        Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".
    '''
    user = f'Normalize time expressions from the following passage which has a document creation time of {dct}: {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

def get_et_ee_prompt(text):
    sys = '''
        You are a temporal linking system. Given a passage with annotated events (E#) and times (T#), link them and infer temporal relations.
        Event-Time relations should be outputed as ["E#", "T#"], if no time can be linked, infer its time indirectly using event-event relations.
        Event-Event temporal relations should be outputed as ["E#", "BEFORE|AFTER|OVERLAPS|CONTAINS|EQUALS|INDENTITY", "E#"] with each event pair appears at most once.
        Return ONLY valid JSON (no markdown, no commentary). 
    '''
    user = f'Link the events to times or events to events from the following passage: {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

def get_tkg_prompt(text, dct):
    #text = " ".join([wrd for sent in text for wrd in sent])
    sys = '''
        You are an information extraction system for geoscience and general texts.
        Extract (1) quintuples (events), and (3) temporal-relation triples.
        Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".

        Quintuples (events)
        - Each event is one quintuple: ["E#", "subject or null", "event string", "object or null", "event start time normalised", "event end time normalised"]
        - Event string must be short (just the trigger words).
        - If the start or end time is in the geological timescale use millions of years (Ma)
        - E# assigned in order of first mention; reuse IDs for duplicates.

        Temporal triples
        - BEFORE: event1 ends before event2 starts
        - AFTER:  event1 starts after event2 ends
        - DURING: event1 occurs fully within event2
        - CONTAINS: event1 fully contains event2
        - IDENTITY/EQUALS: same event/time span
        - OVERLAPS: partial intersection
        - Each relation is ["E#", "BEFORE|AFTER|DURING|CONTAINS|IDENTITY|EQUALS|OVERLAPS", "E#"]
        - Only E# allowed, never T#.
        - Each unordered event pair appears at most once.

        Validation
        - IDs sequential by first mention (E1, E2 ...; T1, T2 ...).
        - All T# in quintuples must exist in times.
        - JSON must be valid: no trailing commas, no comments.
    '''
    user = f'Extract events and temporal relations from the following passage (document creation time: {dct}):    {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

def jsonify(output):
    try:
        return json.loads(output)
    except json.JSONDecodeError:
        try:
            return json.loads(output+"}")
        except json.JSONDecodeError:
            return None

def inference(model, tok, prompt, max_new_tokens=4000):
    input_prompt = tok.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)
    inputs = tok(input_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=False)
    gen_tokens = out[0, inputs.input_ids.shape[-1]:]
    decodings = tok.decode(gen_tokens, skip_special_tokens=True)
    prediction = decodings.strip(":\n`").lstrip("```json\n").rstrip("\n```")
    return prediction

def norm_preprocess(text, times):
    out = ""
    for sn, sent in enumerate(text):
        sent_times = [t['offset'] for t in times if t['sent_id'] == sn]
        sent_times = sorted(sent_times, key=lambda x: x[0], reverse=True)

        for st, en in sent_times:
            sent.insert(en,"</timex>")
            sent.insert(st,"<timex>")
        out += " ".join(sent) + " "
    return out.strip()

def post_processing(preds):
    out = []
    for fn, pred in enumerate(preds):
        json_out = jsonify(pred['pred'])
        if json_out is None:
            testy = pred['pred'].partition('{')[-1].rpartition('}')[0]
            json_out = jsonify('{'+testy+'}')
        
        if json_out is None or json_out == {}:
            testy = pred['pred'].partition('{')[-1][:-6]
            json_out = jsonify('{'+testy+'}')

        if json_out is None:
            testy = pred['pred'].partition('{')[-1][:-1]
            json_out = jsonify('{'+testy+'}')

        if json_out is None:
            testy = pred['pred'].partition('{')[-1].rpartition(']')[0]
            json_out = jsonify('{'+testy+'}')

        if type(json_out) is list:
            json_out = json_out[0]

        keys = list(json_out.keys())
        keys.remove('times')
        keys.remove('quintuples')
        json_out['triples'] = json_out.pop(keys[0])

        out.append({'text':pred['text'], 'pred':json_out})
    return out

def ner_post_processing(ner_preds):
    poster = []

    for fn, pred in enumerate(ner_preds):
        if fn==121:
            continue
        stripped = pred['pred'].replace('{"entity surface text": ', '[').replace('"type": ', '').replace("}",']').replace('{"entity": ',"[").replace("{","[")
        texty = stripped.partition('[')[-1].rpartition(']')[0]
        pred_json = jsonify("["+texty+"]")
        if pred_json is None:
            print(fn)
            pred_json = texty
        poster.append({'text':pred['text'], 'pred':pred_json})
    return poster

def mark_e_t_in_text(sample):
    instances = [instance for instance in sample['instances'] if not(instance['type']!="EVENT" and instance['id']==0)]
    out = ""
    for sn, sent in enumerate(sample['text']):
        sent_cp = deepcopy(sent)
        sent_instances = [(i['offset'], i['id'], i['type']) for i in instances if i['sent_id'] == sn]
        sent_instances = sorted(sent_instances, key=lambda x: x[0], reverse=True)

        for (st, en), id, ty in sent_instances:
            if ty == "EVENT":
                sent_cp.insert(en,f"[/E{id}]")
                sent_cp.insert(st,f"[E{id}]")
            else:
                sent_cp.insert(en,f"[/T{id}]")
                sent_cp.insert(st,f"[T{id}]")
        out += " ".join(sent_cp) + " "
    return out.strip()

In [ ]:
test_data = get_test_data("D:\\GeoTKG\\cleandata\\tie\\test.json")

In [ ]:
def run_llama(prediction_type):
    chat_preds = []
    if prediction_type == "tkg":
        file_num = 1
        for example in test_data:
            dct = [inst['value'] for inst in example['instances'] if inst['type'] != "EVENT" and inst['id'] == 0][0]
            if len(example['text'])>50:
                half = int(len(example['text'])/2)
                prompt1 = get_tkg_prompt(example['text'][:half], dct)
                prompt2 = get_tkg_prompt(example['text'][half:], dct)
                prediction = [inference(model, tok, prompt1), inference(model, tok, prompt2)]
            else:
                prompt = get_tkg_prompt(example['text'], dct)
                prediction = inference(model, tok, prompt)
            chat_preds.append({"text":example['text'], "pred":prediction})
            print(f"Processed example {file_num} / {len(test_data)}")
            file_num += 1
    elif prediction_type == "ner":
        print("----- RUNNING NER PREDICTIONS -----")
        file_num = 1
        for example in test_data:
            if len(example['text'])>50:
                half = int(len(example['text'])/2)
                prompt1 = get_ner_prompt(example['text'][:half], "event and time")
                prompt2 = get_ner_prompt(example['text'][half:], "event and time")
                prediction = [inference(model, tok, prompt1, max_new_tokens=1000), inference(model, tok, prompt2, max_new_tokens=1000)]
            else:
                prompt = get_ner_prompt(example['text'], "event and time")
                prediction = inference(model, tok, prompt, max_new_tokens=1000)
            chat_preds.append({"text":example['text'], "pred":prediction})
            print(f"Processed example {file_num} / {len(test_data)}")
            file_num += 1
    elif prediction_type == "norm":
        file_num = 1
        for example in test_data:
            dct = [inst['value'] for inst in example['instances'] if inst['type'] != "EVENT" and inst['id'] == 0][0]
            prompt = get_norm_prompt(norm_preprocess(example['text'], [instance for instance in example['instances'] if instance['type'] != "EVENT" and instance['id'] != 0]), dct)
            prediction = inference(model, tok, prompt, max_new_tokens=500)
            chat_preds.append({"text":example['text'], "pred":prediction})
            print(f"Processed example {file_num} / {len(test_data)}")
            file_num += 1
    return chat_preds

In [ ]:
chat_preds = run_llama("norm")

In [3]:
with open('D:\GeoTKG\cleandata\geological_tkg_test.json', 'r') as file:
    examples = json.load(file)
chat_preds = []
file_num = 1
for example in examples:
    dct = example['dct']
    prompt = get_tkg_prompt(example['text'], dct)
    prediction = inference(model, tok, prompt, max_new_tokens=2500)
    chat_preds.append({"text":file_num, "pred":prediction})
    print(f"Processed example {file_num} / {len(examples)}")
    file_num += 1

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 1 / 8


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 2 / 8


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Processed example 5 / 8


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 6 / 8


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 7 / 8
Processed example 8 / 8


In [8]:
out_write = []
for fn, pred in enumerate(chat_preds):
    text = pred['pred']#.replace('"null"', 'null').replace("null", '\"null\"')
    json_out = json.loads("["+text+"]")
    print(fn)
    quins = {}
    for thing in json_out[0]:
        quins[thing[0]] = {
            "subject": thing[1],
            "event": thing[2],
            "object": thing[3],
            "s_time": thing[4],
            "e_time": thing[5]
        }
    trips = []
    for trip in json_out[1]:
        trips.append([quins[trip[0]]['event'], trip[1], quins[trip[2]]['event']])
    out_write.append({"text":fn, "pred":{"quintuples":list(quins.values()), "triples":trips}})

0
1


JSONDecodeError: Expecting ',' delimiter: line 7 column 1 (char 332)

In [ ]:
with open("llama3-8B-geotkg-preds.json", 'w') as json_file:
    for sample in chat_preds:
        json_file.write(json.dumps(sample)+"\n")

In [10]:
chat_preds[2]['pred'] = x

'[\n  ["E1", "CRAE", "CRAE regional airborne geophysics", null, "1980-01-01T00:00:00", "1980-12-31T23:59:59"],\n  ["E2", "Newcrest", "Newcrest exploration", null, "1990-01-01T00:00:00", "1990-12-31T23:59:59"],\n  ["E3", "Newcrest", "outline significant zinc anomalism", "Edmund Basin", "1990-01-01T00:00:00", "1990-12-31T23:59:59"]\n]\n\n[\n  ["E1", "BEFORE", "E2"],\n  ["E2", "DURING", "E3"]\n]'

In [11]:
x = '[\n  ["E1", "CRAE", "CRAE regional airborne geophysics", null, "1980-01-01T00:00:00", "1980-12-31T23:59:59"],\n  ["E2", "Newcrest", "Newcrest exploration", null, "1990-01-01T00:00:00", "1990-12-31T23:59:59"],\n  ["E3", "Newcrest", "outline significant zinc anomalism", "Edmund Basin", "1990-01-01T00:00:00", "1990-12-31T23:59:59"]\n],\n\n[\n  ["E1", "BEFORE", "E2"],\n  ["E2", "DURING", "E3"]\n]'


In [12]:
print(x)

[
  ["E1", "CRAE", "CRAE regional airborne geophysics", null, "1980-01-01T00:00:00", "1980-12-31T23:59:59"],
  ["E2", "Newcrest", "Newcrest exploration", null, "1990-01-01T00:00:00", "1990-12-31T23:59:59"],
  ["E3", "Newcrest", "outline significant zinc anomalism", "Edmund Basin", "1990-01-01T00:00:00", "1990-12-31T23:59:59"]
],

[
  ["E1", "BEFORE", "E2"],
  ["E2", "DURING", "E3"]
]
